# 🎙️ Grammar Scoring Engine for Spoken Audio — SHL Hiring Assessment 2026

**Task:** predict a continuous MOS Likert *grammar score* (0 – 5) from a 45–60 s speech clip.
**Data:** 769 labelled training clips, 216 test clips. **Metrics:** RMSE and Pearson correlation.

> **Kaggle settings before you press *Run All***
> 1. Accelerator → **GPU (T4 x2 or P100)**  2. Settings → **Internet → On** (needed once to download the free open-source models from Hugging Face).
> Total runtime ≈ 35–60 min on a T4 (almost all of it is the one-off feature extraction, which is cached to `/kaggle/working`).

---
## 📝 Report (approach in brief)

**Core idea.** A *grammar* grade is a property of **what was said** (syntax, tense, agreement, sentence completeness) *and* of **how it was said** (fluency, hesitations, repairs). A single model rarely captures both, so the pipeline fuses four complementary "views" of every clip and blends them with a stacked ensemble.

| View | Free pretrained model | What it captures |
|---|---|---|
| 1. Acoustic-semantic embedding | **OpenAI Whisper large-v3-turbo encoder** (mid + last layer, mean-pooled) | Pronunciation, fluency, prosody *and* linguistic content – Whisper encoders are among the strongest open speech representations |
| 2. Transcript | **Whisper large-v3-turbo** ASR (decoder) | Words → input for all text views |
| 3. Grammatical acceptability | **RoBERTa-base fine-tuned on CoLA** | P(sentence is grammatical) per sentence / 10-word window |
| 4. Language-model fluency | **GPT-2** | Token-level negative log-likelihood / perplexity (ungrammatical text = high surprise) |
| 5. Semantic text embedding | **all-mpnet-base-v2** | Overall language sophistication / topic-independent quality |
| 6. Hand-crafted | librosa + regex | speech rate, pauses, fillers, lexical diversity, clause-complexity markers, repetition, MFCC stats |

**Pipeline architecture**
```
 wav ─► resample 16 kHz mono ─┬─► Whisper encoder ─► mean-pool (mid / last layer) ───────────┐
                              ├─► Whisper decoder ─► transcript ─┬─► CoLA acceptability ──┐   │
                              │                                  ├─► GPT-2 NLL ───────────┤   │
                              │                                  ├─► mpnet embedding ─────┤   │
                              │                                  └─► stats (rate, fillers…)┤  │
                              └─► librosa (pauses, RMS, MFCC) ───────────────────────────┘   │
                                                                                              ▼
        Standardise → PCA(95 % var) → {Ridge, SVR}  |  LightGBM on hand features  →  OOF predictions
                                                                                              ▼
                                   Non-negative linear stacking  →  clip to [0, 5]  →  submission.csv
```
**Validation.** 5-fold × 3-repeat stratified CV (all scalers / PCA are fitted *inside* each fold → no leakage). The stacker is itself evaluated with a second-level CV.
**Training-set RMSE** (compulsory) is printed in the final section, next to the honest cross-validated RMSE / Pearson.

In [ ]:
# ── 0. Imports & configuration ───────────────────────────────────────────────
import os, re, glob, gc, math, time, warnings, pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

import torch
SEED   = 42
SR     = 16_000                                   # all models expect 16 kHz
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
np.random.seed(SEED); torch.manual_seed(SEED)

# Free, open-source models (downloaded from the Hugging Face Hub)
WHISPER_ID = "openai/whisper-large-v3-turbo"      # ASR + speech encoder
COLA_ID    = "textattack/roberta-base-CoLA"       # grammatical-acceptability classifier
LM_ID      = "gpt2"                               # language model for fluency / perplexity
EMB_ID     = "sentence-transformers/all-mpnet-base-v2"

BATCH       = 8        # clips per Whisper batch (lower to 4 if you hit GPU OOM)
SMOKE_TEST  = False    # True → run on the first 24 clips only (2-minute sanity check)
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "./work"
os.makedirs(WORK, exist_ok=True)
print(f"device={DEVICE}  dtype={DTYPE}  torch={torch.__version__}")

## 1 · Data loading & sanity checks

In [ ]:
# ── 1. Locate the competition files automatically ────────────────────────────
hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
ROOT = os.path.dirname(hits[0]) if hits else "."
print("Data root:", ROOT)

train_df = pd.read_csv(f"{ROOT}/train.csv")
test_df  = pd.read_csv(f"{ROOT}/test.csv")
sample_sub = pd.read_csv(f"{ROOT}/sample_submission.csv")

def audio_map(folder):
    """filename -> full path for every .wav below `folder`."""
    return {os.path.basename(p): p for p in glob.glob(f"{folder}/**/*.wav", recursive=True)}

# NB: train/ and test/ are separate folders and may reuse the same file names → keep separate maps
train_map = audio_map(f"{ROOT}/train")
test_map  = audio_map(f"{ROOT}/test")
print(f"train.csv rows={len(train_df)}  wavs found={len(train_map)} | test.csv rows={len(test_df)}  wavs found={len(test_map)}")

missing_tr = [f for f in train_df.filename if f not in train_map]
missing_te = [f for f in test_df.filename  if f not in test_map]
print("missing train audio:", len(missing_tr), "| missing test audio:", len(missing_te))
assert not missing_tr and not missing_te, "Some audio files listed in the CSVs were not found."

if SMOKE_TEST:
    train_df = train_df.sample(24, random_state=SEED).reset_index(drop=True)
    test_df  = test_df.head(8).copy()
y = train_df["label"].values.astype(float)
train_df.head()

In [ ]:
# ── 2. Quick EDA: label distribution ─────────────────────────────────────────
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
sns.countplot(x=train_df.label, ax=ax[0], color="#4C72B0"); ax[0].set_title("Training label distribution")
ax[0].set_xlabel("Grammar score")
ax[1].pie(train_df.label.round().value_counts().sort_index(), labels=sorted(train_df.label.round().unique()),
          autopct="%1.0f%%", startangle=90); ax[1].set_title("Share per rounded score")
plt.tight_layout(); plt.show()
print(train_df.label.describe().round(3).to_string())
print("\nNote: score 0 clips (≈5 %) are typically silent / unintelligible audio → handled by word-count & speech-ratio features.")

## 2 · Feature extraction
### 2.1 Audio-level helpers (preprocessing)
Preprocessing = decode → **mono, 16 kHz float32**. No denoising or trimming is applied on purpose: pauses, hesitations and background noise are themselves informative for grammar/fluency grades. Whisper's own feature extractor creates 30 s log-mel windows; clips longer than 30 s are split into windows and mean-pooled (weighted by the amount of real speech in each window).

In [ ]:
import librosa

def load_audio(path):
    y_, _ = librosa.load(path, sr=SR, mono=True)
    if len(y_) < SR // 2:                          # guard against empty / corrupt files
        y_ = np.pad(y_, (0, SR // 2 - len(y_)))
    return y_.astype(np.float32)

def acoustic_feats(y_):
    """Fluency-related acoustic statistics (silence structure, loudness, MFCCs)."""
    dur  = len(y_) / SR
    rms  = librosa.feature.rms(y=y_)[0]
    iv   = librosa.effects.split(y_, top_db=30)
    gaps = [(iv[i + 1][0] - iv[i][1]) / SR for i in range(len(iv) - 1)]
    sp   = sum(e - s for s, e in iv) / SR if len(iv) else 0.0
    f = dict(dur=dur, rms_mean=float(rms.mean()), rms_std=float(rms.std()),
             speech_ratio=sp / max(dur, 1e-3), n_voiced=len(iv),
             pause_mean=float(np.mean(gaps)) if gaps else 0.0,
             pause_max=float(np.max(gaps)) if gaps else 0.0,
             n_pause_gt05=int(sum(g > 0.5 for g in gaps)), n_pause_gt1=int(sum(g > 1.0 for g in gaps)))
    mf = librosa.feature.mfcc(y=y_, sr=SR, n_mfcc=13)
    for i in range(13):
        f[f"mfcc{i}_m"] = float(mf[i].mean()); f[f"mfcc{i}_s"] = float(mf[i].std())
    return f

def clean_text(t):
    """Normalise whitespace and collapse ASR 'hallucination loops' (same 1-4 word phrase repeated ≥5×)."""
    t = re.sub(r"\s+", " ", t).strip()
    return re.sub(r"\b(\w+(?:\s+\w+){0,3}?)(?:\s+\1\b){4,}", r"\1", t, flags=re.I)

def seg_feats(chunks, dur):
    """Timing features derived from Whisper's segment timestamps."""
    st = []
    for c in chunks:
        s, e = c["timestamp"]; s = 0.0 if s is None else s; e = dur if e is None else e
        st.append((s, e))
    ln   = [e - s for s, e in st]
    gaps = [max(0.0, st[i + 1][0] - st[i][1]) for i in range(len(st) - 1)]
    return dict(asr_nseg=len(st), asr_seglen_mean=float(np.mean(ln)) if ln else 0.0,
                asr_seglen_std=float(np.std(ln)) if ln else 0.0,
                asr_gap_mean=float(np.mean(gaps)) if gaps else 0.0, asr_gap_max=float(np.max(gaps)) if gaps else 0.0,
                asr_gap_frac=float(np.sum(gaps) / max(dur, 1e-3)) if gaps else 0.0,
                asr_cover=float((st[-1][1] - st[0][0]) / max(dur, 1e-3)) if st else 0.0)

### 2.2 Whisper: speech-encoder embeddings **and** transcripts (one model, two uses)

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

def load_whisper():
    model = AutoModelForSpeechSeq2Seq.from_pretrained(WHISPER_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True).to(DEVICE).eval()
    proc  = AutoProcessor.from_pretrained(WHISPER_ID)
    asr   = pipeline("automatic-speech-recognition", model=model, tokenizer=proc.tokenizer,
                     feature_extractor=proc.feature_extractor, torch_dtype=DTYPE, device=DEVICE)
    return model, proc, asr

@torch.no_grad()
def whisper_embed(model, proc, y_, max_win=4):
    """Mean-pooled Whisper-encoder states (≈60 % depth layer and last layer) over 30 s windows."""
    win = 30 * SR
    chunks = [y_[i * win:(i + 1) * win] for i in range(min(max_win, math.ceil(len(y_) / win)))]
    chunks = [c for c in chunks if len(c) >= SR // 2] or [y_]
    feats  = proc.feature_extractor(chunks, sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE, DTYPE)
    hs     = model.get_encoder()(feats, output_hidden_states=True).hidden_states
    valid  = torch.tensor([max(1, int(len(c) / SR * 50)) for c in chunks], dtype=torch.float32, device=DEVICE)  # 50 frames / s
    def pool(h):
        v = torch.stack([h[i, :int(valid[i])].float().mean(0) for i in range(len(chunks))])
        return ((v * valid[:, None]).sum(0) / valid.sum()).cpu().numpy()
    return pool(hs[int((len(hs) - 1) * 0.6)]), pool(hs[-1])

def transcribe(asr, ys):
    kw = dict(batch_size=len(ys), chunk_length_s=30, stride_length_s=5, return_timestamps=True,
              generate_kwargs={"language": "en", "task": "transcribe"})
    try:
        out = asr([{"raw": a, "sampling_rate": SR} for a in ys], **kw)
    except Exception as e:                                   # fall back to one-by-one on any batch failure
        print("batch transcription failed, retrying singly:", repr(e)[:120])
        out = [asr({"raw": a, "sampling_rate": SR}, **{**kw, "batch_size": 1}) for a in ys]
    return out

def extract_split(df, amap, tag, model, proc, asr):
    cache = f"{WORK}/{tag}{'_smoke' if SMOKE_TEST else ''}_raw.pkl"
    if os.path.exists(cache):
        print("loaded cache", cache); return pickle.load(open(cache, "rb"))
    names = df.filename.tolist(); R = dict(names=names, mid=[], last=[], text=[], ac=[], seg=[])
    for i in tqdm(range(0, len(names), BATCH), desc=f"extract {tag}"):
        ys = [load_audio(amap[n]) for n in names[i:i + BATCH]]
        for a in ys:
            m, l = whisper_embed(model, proc, a); R["mid"].append(m); R["last"].append(l); R["ac"].append(acoustic_feats(a))
        for a, r in zip(ys, transcribe(asr, ys)):
            R["text"].append(clean_text(r.get("text", ""))); R["seg"].append(seg_feats(r.get("chunks", []), len(a) / SR))
    R["mid"], R["last"] = np.vstack(R["mid"]), np.vstack(R["last"])
    pickle.dump(R, open(cache, "wb")); return R

t0 = time.time()
whisper, wproc, wasr = load_whisper()
RAW_TR = extract_split(train_df, train_map, "train", whisper, wproc, wasr)
RAW_TE = extract_split(test_df,  test_map,  "test",  whisper, wproc, wasr)
del whisper, wproc, wasr; gc.collect(); torch.cuda.empty_cache()
print(f"Whisper stage done in {(time.time()-t0)/60:.1f} min | enc dim = {RAW_TR['mid'].shape[1]}")
pd.DataFrame({"filename": RAW_TR["names"][:5], "label": y[:5], "transcript": [t[:230] for t in RAW_TR["text"][:5]]})

### 2.3 Text-based grammar features
* **CoLA acceptability** – a RoBERTa classifier trained on the *Corpus of Linguistic Acceptability*; we score every sentence **and** every 10-word window (Whisper punctuation is imperfect, so windows give a punctuation-independent view).
* **GPT-2 surprisal** – grammatical, fluent English has lower per-token negative log-likelihood.
* **Hand-crafted stats** – speech rate, lexical diversity, filler words, repetition, and clause-complexity markers (subordinators, perfect/modal auxiliaries) that separate score 3 from 5 in the rubric.
* **mpnet sentence embedding** – a topic-agnostic semantic summary of the transcript.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM, AutoModel

FILLERS = {"um", "uh", "uhm", "er", "erm", "hmm", "ah", "mm"}
SUBORD  = {"because", "although", "though", "which", "that", "if", "when", "while", "whereas", "since", "unless",
           "however", "therefore", "moreover", "who", "whom", "whose", "where", "after", "before", "until"}
AUX     = {"have", "has", "had", "been", "being", "would", "could", "should", "might", "must", "will", "shall"}

def split_sents(t): return [s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if len(s.split()) >= 3]
def windows(t, n=10):
    w = t.split(); return [" ".join(w[i:i + n]) for i in range(0, max(len(w) - n + 1, 1), n) if len(w[i:i + n]) >= 4]

def stat_feats(text, dur):
    words = re.findall(r"[A-Za-z']+", text.lower()); nw = len(words); sents = split_sents(text)
    bi = list(zip(words, words[1:])); tri = list(zip(words, words[1:], words[2:]))
    return dict(n_words=nw, wps=nw / max(dur, 1), n_sents=len(sents), avg_sent_len=nw / max(len(sents), 1),
                ttr=len(set(words)) / max(nw, 1), avg_word_len=float(np.mean([len(w) for w in words])) if words else 0.0,
                long_word_frac=sum(len(w) >= 7 for w in words) / max(nw, 1), n_fillers=sum(w in FILLERS for w in words),
                subord_rate=sum(w in SUBORD for w in words) / max(nw, 1), aux_rate=sum(w in AUX for w in words) / max(nw, 1),
                rep_bigram=1 - len(set(bi)) / max(len(bi), 1), rep_trigram=1 - len(set(tri)) / max(len(tri), 1),
                comma_rate=text.count(",") / max(nw, 1), unique_words=len(set(words)))

def safe_load(fn, name):
    try: return fn()
    except Exception as e:
        print(f"⚠ could not load {name}: {repr(e)[:150]}  → that feature block is skipped"); return None

@torch.no_grad()
def cola_feats(texts, bs=32):
    tok = AutoTokenizer.from_pretrained(COLA_ID); mdl = AutoModelForSequenceClassification.from_pretrained(COLA_ID).to(DEVICE).eval()
    units, owner, kind = [], [], []
    for d, t in enumerate(texts):
        for u in split_sents(t): units.append(u); owner.append(d); kind.append("s")
        for u in windows(t):     units.append(u); owner.append(d); kind.append("w")
    p = []
    for i in tqdm(range(0, len(units), bs), desc="CoLA"):
        enc = tok(units[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=64).to(DEVICE)
        p += torch.softmax(mdl(**enc).logits.float(), -1)[:, 1].cpu().tolist()       # P(acceptable)
    df = pd.DataFrame(dict(d=owner, k=kind, p=p)); rows = []
    for d in range(len(texts)):
        f = {}
        for k in ("s", "w"):
            v = df[(df.d == d) & (df.k == k)].p.values
            f.update({f"cola_{k}_mean": v.mean() if len(v) else 0.5, f"cola_{k}_min": v.min() if len(v) else 0.5,
                      f"cola_{k}_std": v.std() if len(v) > 1 else 0.0, f"cola_{k}_ok": (v > 0.5).mean() if len(v) else 0.5})
        rows.append(f)
    del mdl; torch.cuda.empty_cache(); return pd.DataFrame(rows)

@torch.no_grad()
def lm_feats(texts):
    tok = AutoTokenizer.from_pretrained(LM_ID); mdl = AutoModelForCausalLM.from_pretrained(LM_ID).to(DEVICE).eval(); rows = []
    for t in tqdm(texts, desc="GPT-2"):
        ids = tok(t if t.strip() else "...", return_tensors="pt", truncation=True, max_length=512).input_ids.to(DEVICE)
        if ids.shape[1] < 2: ids = torch.cat([ids, ids], 1)
        lp = torch.log_softmax(mdl(ids).logits[0, :-1].float(), -1)
        nll = (-lp.gather(1, ids[0, 1:, None])[:, 0]).cpu().numpy()
        rows.append(dict(lm_nll_mean=nll.mean(), lm_nll_std=nll.std(), lm_nll_p90=np.percentile(nll, 90),
                         lm_nll_median=np.median(nll), lm_ppl=float(np.exp(min(nll.mean(), 20)))))
    del mdl; torch.cuda.empty_cache(); return pd.DataFrame(rows)

@torch.no_grad()
def mpnet_emb(texts, bs=32):
    tok = AutoTokenizer.from_pretrained(EMB_ID); mdl = AutoModel.from_pretrained(EMB_ID).to(DEVICE).eval(); out = []
    for i in tqdm(range(0, len(texts), bs), desc="mpnet"):
        enc = tok([t if t.strip() else "." for t in texts[i:i + bs]], return_tensors="pt", padding=True, truncation=True, max_length=384).to(DEVICE)
        h = mdl(**enc).last_hidden_state; m = enc.attention_mask.unsqueeze(-1).float()
        e = (h * m).sum(1) / m.sum(1); out.append(torch.nn.functional.normalize(e, dim=-1).cpu().numpy())
    del mdl; torch.cuda.empty_cache(); return np.vstack(out)

def build_blocks(R, df):
    """Turn the raw extraction dict into named feature blocks."""
    texts = R["text"]; dur = [a["dur"] for a in R["ac"]]
    parts = [pd.DataFrame(R["ac"]), pd.DataFrame(R["seg"]), pd.DataFrame([stat_feats(t, d) for t, d in zip(texts, dur)])]
    c = safe_load(lambda: cola_feats(texts), "CoLA");  parts += [c] if c is not None else []
    l = safe_load(lambda: lm_feats(texts), "GPT-2");   parts += [l] if l is not None else []
    hand = pd.concat(parts, axis=1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    B = {"hand": hand, "enc_mid": R["mid"], "enc_last": R["last"]}
    e = safe_load(lambda: mpnet_emb(texts), "mpnet")
    if e is not None: B["txt_emb"] = e
    return B

BL_TR = build_blocks(RAW_TR, train_df); BL_TE = build_blocks(RAW_TE, test_df)
assert list(BL_TR["hand"].columns) == list(BL_TE["hand"].columns)
print({k: np.shape(v) for k, v in BL_TR.items()})

### 2.4 Exploratory look at the engineered features

In [ ]:
H = BL_TR["hand"].copy(); H["label"] = y
corr = H.corr(numeric_only=True)["label"].drop("label").sort_values(key=np.abs, ascending=False)
top = corr.head(15)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
sns.barplot(x=top.values, y=top.index, ax=ax[0], palette="vlag"); ax[0].set_title("Top-15 features by Pearson r with label")
sns.scatterplot(x=H["n_words"], y=H["label"], alpha=.5, ax=ax[1]); ax[1].set_title("Transcript length vs. grammar score")
col = "cola_w_mean" if "cola_w_mean" in H else "ttr"
sns.boxplot(x=H["label"].round(), y=H[col], ax=ax[2], color="#8FB8DE"); ax[2].set_title(f"{col} per grammar score")
plt.tight_layout(); plt.show()

## 3 · Modelling
Each feature block feeds small, strongly-regularised learners (n = 769 is tiny, so *simple + diverse* beats a single large model):

* **Embedding blocks** → `StandardScaler → PCA(95 % variance) → Ridge` and `→ SVR (RBF)`
* **Hand-crafted block** → `LightGBM`, `SVR`, `Ridge`
* **"all"** (Whisper-mid + mpnet + hand-crafted) → `PCA → SVR / Ridge`

All preprocessing lives inside sklearn pipelines so it is re-fitted per fold (no leakage). Out-of-fold (OOF) predictions are blended by a **non-negative linear stacker**.

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.svm import SVR
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import lightgbm as lgb

rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, b)))
pear = lambda a, b: float(pearsonr(a, b)[0])

def emb_ridge(v=0.95): return make_pipeline(StandardScaler(), PCA(v, svd_solver="full", random_state=SEED), RidgeCV(alphas=np.logspace(0, 4, 13)))
def emb_svr(v=0.95, C=2.0): return make_pipeline(StandardScaler(), PCA(v, svd_solver="full", random_state=SEED), SVR(C=C, epsilon=0.2, gamma="scale"))
def hand_svr():  return make_pipeline(StandardScaler(), SVR(C=2.0, epsilon=0.2, gamma="scale"))
def hand_ridge(): return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 13)))
def hand_lgbm(): return lgb.LGBMRegressor(n_estimators=500, learning_rate=0.02, num_leaves=8, min_child_samples=15, subsample=0.8,
                                         subsample_freq=1, colsample_bytree=0.6, reg_lambda=5.0, random_state=SEED, verbose=-1, n_jobs=-1)

# assemble the concatenated "all" block
def make_all(B): return np.hstack([B["enc_mid"], B["txt_emb"] if "txt_emb" in B else np.zeros((len(B["hand"]), 0)), B["hand"].values])
XB_TR = {k: (v.values if isinstance(v, pd.DataFrame) else v) for k, v in BL_TR.items()}
XB_TE = {k: (v.values if isinstance(v, pd.DataFrame) else v) for k, v in BL_TE.items()}
XB_TR["all"], XB_TE["all"] = make_all(BL_TR), make_all(BL_TE)

ZOO = {"enc_mid_ridge": ("enc_mid", emb_ridge), "enc_mid_svr": ("enc_mid", emb_svr),
       "enc_last_ridge": ("enc_last", emb_ridge), "enc_last_svr": ("enc_last", emb_svr),
       "hand_lgbm": ("hand", hand_lgbm), "hand_svr": ("hand", hand_svr), "hand_ridge": ("hand", hand_ridge),
       "all_ridge": ("all", emb_ridge), "all_svr": ("all", emb_svr)}
if "txt_emb" in XB_TR:
    ZOO.update({"txt_emb_ridge": ("txt_emb", emb_ridge), "txt_emb_svr": ("txt_emb", emb_svr)})
print("Base learners:", list(ZOO))

In [ ]:
# ── Cross-validated training of every base learner ───────────────────────────
N_SPLITS, N_REPEATS = (3, 1) if SMOKE_TEST else (5, 3)
strat = np.clip(np.round(y), 1, 5).astype(int)            # merge the tiny '0/1' strata for stratified splitting

OOF, TEST, CVSTAT = {}, {}, []
for name, (blk, factory) in tqdm(ZOO.items(), desc="CV base models"):
    Xtr, Xte = XB_TR[blk], XB_TE[blk]
    oof_r = np.zeros((N_REPEATS, len(y))); tp = np.zeros(len(Xte)); rm, pr = [], []
    for r in range(N_REPEATS):
        for tr, va in StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED + r).split(Xtr, strat):
            m = factory().fit(Xtr[tr], y[tr])
            oof_r[r, va] = m.predict(Xtr[va]); tp += m.predict(Xte) / (N_SPLITS * N_REPEATS)
        rm.append(rmse(y, np.clip(oof_r[r], 0, 5))); pr.append(pear(y, oof_r[r]))
    OOF[name], TEST[name] = oof_r.mean(0), tp
    CVSTAT.append(dict(model=name, block=blk, cv_rmse=np.mean(rm), cv_rmse_std=np.std(rm), cv_pearson=np.mean(pr)))
cv_table = pd.DataFrame(CVSTAT).sort_values("cv_rmse").reset_index(drop=True)
cv_table.round(4)

In [ ]:
# ── Stacking: non-negative linear blend of OOF predictions ───────────────────
names = list(ZOO); O = np.column_stack([OOF[n] for n in names]); T = np.column_stack([TEST[n] for n in names])
stacker = LinearRegression(positive=True).fit(O, y)
w = pd.Series(stacker.coef_, index=names).sort_values(ascending=False)
print("Stack weights (intercept = %.3f):" % stacker.intercept_); print(w.round(3).to_string())

# honest estimate of the stacker: second-level CV on the OOF matrix
oof_stack = cross_val_predict(LinearRegression(positive=True), O, y, cv=KFold(5, shuffle=True, random_state=0))
oof_stack = np.clip(oof_stack, 0, 5)
CV_RMSE, CV_PEAR = rmse(y, oof_stack), pear(y, oof_stack)
print(f"\nStacked ensemble  CV-RMSE = {CV_RMSE:.4f}   CV-Pearson = {CV_PEAR:.4f}")
print(f"Best single model CV-RMSE = {cv_table.cv_rmse.min():.4f}  ({cv_table.model[0]})")

### 3.1 Training-set RMSE (compulsory) — all base learners re-fitted on 100 % of the training data

In [ ]:
full_models, TRAIN_P, TEST_P = {}, [], []
for n in names:
    blk, factory = ZOO[n]; m = factory().fit(XB_TR[blk], y); full_models[n] = m
    TRAIN_P.append(m.predict(XB_TR[blk])); TEST_P.append(m.predict(XB_TE[blk]))
train_pred = np.clip(stacker.predict(np.column_stack(TRAIN_P)), 0, 5)

TRAIN_RMSE, TRAIN_PEAR = rmse(y, train_pred), pear(y, train_pred)
print("=" * 62)
print(f" TRAINING-SET  RMSE     : {TRAIN_RMSE:.4f}   (in-sample, models fitted on all 769 clips)")
print(f" TRAINING-SET  Pearson  : {TRAIN_PEAR:.4f}")
print(f" CROSS-VALIDATED RMSE   : {CV_RMSE:.4f}   <- honest estimate of test error")
print(f" CROSS-VALIDATED Pearson: {CV_PEAR:.4f}")
print("=" * 62)

## 4 · Evaluation visuals

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(19, 10.5))

# (a) model comparison
ct = cv_table.sort_values("cv_rmse", ascending=False)
ax[0, 0].barh(ct.model, ct.cv_rmse, xerr=ct.cv_rmse_std, color="#6A9FD1"); ax[0, 0].axvline(CV_RMSE, color="crimson", ls="--", label=f"stack {CV_RMSE:.3f}")
ax[0, 0].axvline(y.std(), color="grey", ls=":", label=f"baseline (std) {y.std():.3f}"); ax[0, 0].set_xlabel("CV RMSE (lower = better)"); ax[0, 0].set_title("Base learners vs. stacked ensemble"); ax[0, 0].legend()

# (b) OOF scatter
jit = np.random.RandomState(0).normal(0, .04, len(y))
ax[0, 1].scatter(y + jit, oof_stack, alpha=.45, s=18); ax[0, 1].plot([0, 5], [0, 5], "r--")
ax[0, 1].set(xlabel="True score", ylabel="Out-of-fold prediction", title=f"OOF predictions  (r = {CV_PEAR:.3f}, RMSE = {CV_RMSE:.3f})")

# (c) residuals
res = oof_stack - y; sns.histplot(res, kde=True, ax=ax[0, 2], color="#55A868"); ax[0, 2].axvline(0, color="k")
ax[0, 2].set(title=f"OOF residuals (mean {res.mean():+.3f}, sd {res.std():.3f})", xlabel="prediction − truth")

# (d) error per true score
err = pd.DataFrame(dict(true=y, abs_err=np.abs(res))); sns.boxplot(x=err.true, y=err.abs_err, ax=ax[1, 0], color="#E8B07A")
ax[1, 0].set(title="Absolute error by true grammar score", xlabel="True score", ylabel="|error|")

# (e) train vs test prediction distribution
final_test = np.clip(stacker.predict(T), 0, 5)
sns.kdeplot(y, ax=ax[1, 1], label="train truth", fill=True, alpha=.3); sns.kdeplot(final_test, ax=ax[1, 1], label="test prediction", fill=True, alpha=.3)
ax[1, 1].set(title="Label vs. test-prediction distribution", xlabel="score"); ax[1, 1].legend()

# (f) LightGBM feature importance
lg = full_models["hand_lgbm"]; imp = pd.Series(lg.feature_importances_, index=BL_TR["hand"].columns).sort_values().tail(15)
ax[1, 2].barh(imp.index, imp.values, color="#8172B2"); ax[1, 2].set_title("LightGBM top-15 hand-crafted features (split count)")
plt.tight_layout(); plt.show()

# rounded-score confusion view
cm = pd.crosstab(np.round(y * 1).clip(0, 5), np.round(oof_stack).clip(0, 5), rownames=["true"], colnames=["pred (rounded)"])
plt.figure(figsize=(6, 4.5)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues"); plt.title("Rounded OOF predictions vs truth"); plt.show()

## 5 · Create `submission.csv`

In [ ]:
# ── Build the submission in the competition format (filename,label) ──────────
pred_map = dict(zip(RAW_TE["names"], np.clip(stacker.predict(T), 0, 5)))
sub = pd.DataFrame({"filename": test_df.filename, "label": test_df.filename.map(pred_map).round(4)})
assert sub.label.notna().all() and sub.label.between(0, 5).all() and len(sub) == len(test_df)
sub.to_csv(f"{WORK}/submission.csv", index=False)

# Diagnostic: how does our file compare with the provided sample_submission.csv ?
same = set(sub.filename) == set(sample_sub.filename)
print(f"submission rows: {len(sub)} | sample_submission rows: {len(sample_sub)} | identical file names: {same}")
if not same:
    print(f"⚠ {len(set(sample_sub.filename) - set(sub.filename))} names in sample_submission are not in test.csv. "
          "We follow test.csv (the official list of the 216 test clips). If Kaggle rejects the file, check the 'test' folder names.")
sub.head(10)

In [ ]:
# ── Final report numbers ─────────────────────────────────────────────────────
from IPython.display import Markdown, display
display(Markdown(f"""
### ✅ Results summary
| Metric | Value |
|---|---|
| **Training-set RMSE** (in-sample, required) | **{TRAIN_RMSE:.4f}** |
| Training-set Pearson | {TRAIN_PEAR:.4f} |
| **5-fold × 3 CV RMSE** (stacked, honest) | **{CV_RMSE:.4f}** |
| CV Pearson | {CV_PEAR:.4f} |
| Naive baseline RMSE (predict the mean) | {y.std():.4f} |
| Best single base learner (CV RMSE) | {cv_table.model[0]} – {cv_table.cv_rmse.min():.4f} |

**Interpretation.** The training RMSE is optimistic (flexible learners such as SVR/LightGBM fit the training set closely); the cross-validated figure is the realistic
expectation for the 216 hidden test clips. Whisper-encoder embeddings carry most of the signal, the CoLA / GPT-2 / fluency features add complementary,
interpretable grammar evidence, and stacking removes individual-model variance.

**Possible further gains:** fine-tune Whisper-encoder top layers with a regression head, add `wavlm-large`/`wav2vec2` embeddings as another view, or add a quantised
instruction-LLM "grammar-grade" zero-shot score on the transcript as an extra stacking input.
"""))